### AlexNet Implementation in Pytorch


[Research Paper Link](https://proceedings.neurips.cc/paper_files/paper/2012/file/c399862d3b9d6b76c8436e924a68c45b-Paper.pdf)

![AlexNet Architecture](https://raw.githubusercontent.com/blurred-machine/Data-Science/master/Deep%20Learning%20SOTA/img/alexnet2.png)

### Imports


In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
import matplotlib.pyplot as plt
import numpy as np
from torch.utils.data import DataLoader

### Dataset Image Transformations

In [2]:
# Define transformations for the dataset
transform = transforms.Compose(
    [
        transforms.Resize((224, 224)),  # Resize to match AlexNet input size
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ]
)

### Loading the Dataset

In [3]:
# Load Flowers dataset (Assuming it's available in a directory structure)
dataset_path = "../../data/flowers"
train_dataset = torchvision.datasets.ImageFolder(
    root=dataset_path + "/train", transform=transform
)
test_dataset = torchvision.datasets.ImageFolder(
    root=dataset_path + "/test", transform=transform
)

In [4]:
# Create DataLoaders
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

### Dataset Information

In [5]:
# Get dataset information
def dataset_info(loader, dataset_name="Dataset"):
    print(f"Dataset: {dataset_name}")
    print(f"Total samples: {len(loader.dataset)}")  # Total number of images

    # Get a single batch of data
    images, labels = next(iter(loader))
    print(f"Batch size: {images.shape[0]}")
    print(
        f"Image shape: {images.shape}"
    )  # Shape of images (batch_size, channels, height, width)
    print(f"Unique classes: {set(labels.numpy())}")  # Unique labels in the batch
    print(f"Total classes: {len(set(labels.numpy()))}")  # Number of classes


# Call the function for train and test sets
dataset_info(train_loader, "Training Set")

Dataset: Training Set
Total samples: 1275
Batch size: 32
Image shape: torch.Size([32, 3, 224, 224])
Unique classes: {np.int64(0), np.int64(1)}
Total classes: 2


### AlexNet Model Architecture

In [6]:
# Define the AlexNet model
class AlexNet(nn.Module):
    def __init__(self, num_classes=2):
        super(AlexNet, self).__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 64, kernel_size=11, stride=4, padding=2),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=3, stride=2),
            nn.Conv2d(64, 192, kernel_size=5, padding=2),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=3, stride=2),
            nn.Conv2d(192, 384, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(384, 256, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(256, 256, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=3, stride=2),
        )
        self.classifier = nn.Sequential(
            nn.Dropout(),
            nn.Linear(256 * 6 * 6, 4096),
            nn.ReLU(inplace=True),
            nn.Dropout(),
            nn.Linear(4096, 4096),
            nn.ReLU(inplace=True),
            nn.Linear(4096, num_classes),
        )

    def forward(self, x):
        x = self.features(x)
        x = torch.flatten(x, 1)
        x = self.classifier(x)
        return x